# SellerPulse — SQL Analysis Layer
*Re-expresses the full activation & churn analysis in SQL (SQLite), validating it against the pandas results.*

**Why this notebook exists:** the main analysis was built in Python/pandas. This notebook rebuilds the same logic in **SQL** — funnel, cohorts, the health-score signals, and at-risk flagging — using CTEs, window functions, and conditional aggregation. Running both and getting matching numbers is itself a validation story worth mentioning in an interview.

**Setup:** same as the main notebook — mount Drive or upload the Olist CSVs, then run top to bottom. SQLite ships with Python, so there's nothing to install.

## Step 1 — Load the CSVs into an in-memory SQLite database
We read each CSV with pandas, then push it into SQLite as a table. From there, everything is pure SQL.

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/SellerPulse")   # change if needed

con = sqlite3.connect(":memory:")   # in-memory DB; nothing written to disk

tables = {
    "orders":   "olist_orders_dataset.csv",
    "items":    "olist_order_items_dataset.csv",
    "sellers":  "olist_sellers_dataset.csv",
    "reviews":  "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "cat":      "product_category_name_translation.csv",
}
for tbl, fname in tables.items():
    pd.read_csv(DATA_DIR / fname).to_sql(tbl, con, if_exists="replace", index=False)

def q(sql):
    """Run a SQL query and return a DataFrame (for nice display)."""
    return pd.read_sql_query(sql, con)

print("Loaded tables:", list(tables.keys()))
q("SELECT name FROM sqlite_master WHERE type='table'")

## Step 2 — Build a clean `sale_events` view
One row per (seller, order) = one sale. We attach the purchase date, rank each seller's sales in order, and compute days since their first sale. This view is the backbone every later query builds on.

**SQL concepts here:** `JOIN`, `GROUP BY`, and the window functions `ROW_NUMBER()` and `MIN() OVER (...)`.

In [ ]:
con.executescript("""
DROP VIEW IF EXISTS sale_events;
CREATE VIEW sale_events AS
WITH sales AS (
    SELECT  i.seller_id,
            i.order_id,
            o.order_purchase_timestamp AS sale_ts
    FROM items i
    JOIN orders o ON o.order_id = i.order_id
    WHERE o.order_status <> 'unavailable'
      AND o.order_purchase_timestamp IS NOT NULL
    GROUP BY i.seller_id, i.order_id          -- collapse multi-item orders to one sale
),
ranked AS (
    SELECT  seller_id,
            order_id,
            DATE(sale_ts) AS sale_date,
            ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY sale_ts) AS sale_rank,
            DATE(MIN(sale_ts) OVER (PARTITION BY seller_id)) AS first_sale_date
    FROM sales
)
SELECT  seller_id,
        order_id,
        sale_date,
        sale_rank,
        first_sale_date,
        CAST(julianday(sale_date) - julianday(first_sale_date) AS INT) AS days_since_first
FROM ranked;
""")
q("SELECT * FROM sale_events ORDER BY seller_id, sale_rank LIMIT 8")

## Step 3 — The cohort base (handle observation censoring)
A seller who joined 10 days before the data ends can't show 90-day retention. So we keep only sellers with **≥90 days of possible observation** — same rule as the pandas version, so churn isn't overstated.

**SQL concepts:** subquery for the global max date, `HAVING`-style filtering via a CTE.

In [ ]:
con.executescript("""
DROP VIEW IF EXISTS cohort_base;
CREATE VIEW cohort_base AS
WITH bounds AS (
    SELECT MAX(sale_date) AS dataset_end FROM sale_events
),
per_seller AS (
    SELECT  se.seller_id,
            se.first_sale_date,
            MAX(se.sale_date) AS last_sale_date,
            COUNT(DISTINCT se.order_id) AS total_sales,
            CAST(julianday((SELECT dataset_end FROM bounds)) - julianday(se.first_sale_date) AS INT) AS observable_days
    FROM sale_events se
    GROUP BY se.seller_id, se.first_sale_date
)
SELECT * FROM per_seller
WHERE observable_days >= 90;
""")
q("SELECT COUNT(*) AS sellers_in_base, MIN(observable_days) AS min_obs FROM cohort_base")

## Step 4 — Milestone flags + the activation funnel
For each seller in the base, flag whether they hit each milestone. This uses **conditional aggregation** — `MAX(CASE WHEN ... THEN 1 ELSE 0 END)` — the single most useful SQL pattern for funnels.

`# DECISION:` headline churn = did NOT reach a 2nd sale within 30 days.

In [ ]:
con.executescript("""
DROP VIEW IF EXISTS seller_flags;
CREATE VIEW seller_flags AS
SELECT
    cb.seller_id,
    cb.first_sale_date,
    cb.last_sale_date,
    cb.total_sales,
    cb.observable_days,
    MAX(CASE WHEN se.sale_rank >= 2 AND se.days_since_first <= 30 THEN 1 ELSE 0 END) AS second_sale_30d,
    MAX(CASE WHEN se.days_since_first > 30 AND se.days_since_first <= 90 THEN 1 ELSE 0 END) AS active_30_90,
    MAX(CASE WHEN se.sale_rank >= 5 THEN 1 ELSE 0 END) AS fifth_sale
FROM cohort_base cb
JOIN sale_events se ON se.seller_id = cb.seller_id
GROUP BY cb.seller_id, cb.first_sale_date, cb.last_sale_date, cb.total_sales, cb.observable_days;
""")

q("""
SELECT 'Onboarded (1st sale)' AS stage, COUNT(*) AS sellers FROM seller_flags
UNION ALL
SELECT '2nd sale within 30d', SUM(second_sale_30d) FROM seller_flags
UNION ALL
SELECT 'Active in days 30-90', SUM(active_30_90) FROM seller_flags
UNION ALL
SELECT 'Reached 5th sale', SUM(fifth_sale) FROM seller_flags
""")

In [ ]:
# Headline churn number (should match the pandas notebook: ~40.5%)
q("""
SELECT ROUND(100.0 * (1 - AVG(second_sale_30d)), 1) AS churn_before_2nd_pct
FROM seller_flags
""")

## Step 5 — Cohort retention by join month
Group sellers by the month of their first sale, then show the % hitting each milestone per cohort. **SQL concepts:** `strftime` for month bucketing, `AVG` of the 1/0 flags as a rate.

In [ ]:
q("""
SELECT  strftime('%Y-%m', first_sale_date) AS cohort_month,
        COUNT(*) AS sellers,
        ROUND(100.0 * AVG(second_sale_30d), 1) AS pct_2nd_30d,
        ROUND(100.0 * AVG(active_30_90), 1)   AS pct_active_30_90,
        ROUND(100.0 * AVG(fifth_sale), 1)     AS pct_5th
FROM seller_flags
GROUP BY cohort_month
ORDER BY cohort_month
LIMIT 12
""")

## Step 6 — Worst dropoff categories
Attach each seller's **primary category** (the category of most of their item lines), then compute churn per category. `# DECISION:` require ≥30 sellers so a tiny category can't top the list on noise.

**SQL concepts:** a window function to pick each seller's top category, then aggregation + `HAVING`.

In [ ]:
q("""
WITH item_cat AS (
    SELECT  i.seller_id,
            COALESCE(c.product_category_name_english, p.product_category_name, 'unknown') AS category
    FROM items i
    LEFT JOIN products p ON p.product_id = i.product_id
    LEFT JOIN cat c ON c.product_category_name = p.product_category_name
),
ranked_cat AS (
    SELECT  seller_id, category,
            ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY COUNT(*) DESC) AS rn
    FROM item_cat
    GROUP BY seller_id, category
),
primary_cat AS (
    SELECT seller_id, category FROM ranked_cat WHERE rn = 1
)
SELECT  pc.category,
        COUNT(*) AS sellers,
        ROUND(100.0 * (1 - AVG(sf.second_sale_30d)), 1) AS churn_before_2nd
FROM seller_flags sf
JOIN primary_cat pc ON pc.seller_id = sf.seller_id
GROUP BY pc.category
HAVING COUNT(*) >= 30
ORDER BY churn_before_2nd DESC
LIMIT 10
""")

## Step 7 — Health-score signals in SQL
Compute the four raw signals per seller: velocity, delivery SLA, average review, recency. (The final 0–100 normalization + weighting is easier in pandas, but here are the underlying signals in SQL.)

**SQL concepts:** multiple `LEFT JOIN`s to subqueries, date math, conditional aggregation for on-time rate.

In [ ]:
q("""
WITH bounds AS (SELECT MAX(sale_date) AS dataset_end FROM sale_events),
delivery AS (
    SELECT  i.seller_id,
            AVG(CASE WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date
                     THEN 1.0 ELSE 0.0 END) AS delivery_sla
    FROM items i
    JOIN orders o ON o.order_id = i.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
    GROUP BY i.seller_id
),
review AS (
    SELECT  i.seller_id, AVG(r.review_score) AS avg_review
    FROM items i
    JOIN reviews r ON r.order_id = i.order_id
    GROUP BY i.seller_id
)
SELECT  cb.seller_id,
        ROUND(cb.total_sales * 7.0 / MAX(cb.observable_days, 7), 3) AS velocity_per_week,
        ROUND(d.delivery_sla, 3) AS delivery_sla,
        ROUND(v.avg_review, 2)   AS avg_review,
        CAST(julianday((SELECT dataset_end FROM bounds)) - julianday(cb.last_sale_date) AS INT) AS days_since_last
FROM cohort_base cb
LEFT JOIN delivery d ON d.seller_id = cb.seller_id
LEFT JOIN review   v ON v.seller_id = cb.seller_id
LIMIT 10
""")

## Step 8 — Validation: do the SQL numbers match pandas?
The headline should be **~40.5% churn** and **~2,574 sellers in the base** — identical to the pandas notebook. Matching results across two independent implementations is a strong correctness signal.

> **Interview line:** *"I built the analysis in pandas and independently re-derived the funnel and cohorts in SQL; the numbers matched, which validated the logic."*

---
### SQL techniques demonstrated in this notebook
CTEs (`WITH`), window functions (`ROW_NUMBER`, `MIN() OVER`), conditional aggregation (`MAX(CASE WHEN...)`), date math (`julianday`, `strftime`), multi-table `JOIN`s, `HAVING` filters, and rate calculations via `AVG` of 1/0 flags — the core toolkit analyst interviews test.